<a href="https://colab.research.google.com/github/tinawanggg/Wang_DSPN_S26/blob/master/book/exercises/10_mixed-effects-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [4]:
# WRITE YOUR CODE HERE
library(tidyverse)

lexical_data <- read_csv("LexicalData.csv")
items <- read_csv("Items.csv")

lexical_data <- lexical_data %>%
  mutate(D_RT = as.numeric(gsub(",", "", D_RT)))

lexical_data <- lexical_data %>%
  left_join(items %>% select(Word, Length, Log_Freq_HAL), by = c("D_Word" = "Word"))

Rows: 62610 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): D_Word
dbl (4): Sub_ID, Trial, Type, D_Zscore
num (1): D_RT
lgl (1): Outlier

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 30959 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): Word
dbl (3): Occurrences, Length, Log_Freq_HAL
num (1): Freq_HAL

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [7]:
# WRITE YOUR CODE HERE
linear_model <- lm(D_RT ~ Log_Freq_HAL * Length, data = lexical_data)

summary(linear_model)


Call:
lm(formula = D_RT ~ Log_Freq_HAL * Length, data = lexical_data)

Residuals:
     Min       1Q   Median       3Q      Max 
-1118.01  -205.23   -86.95    90.77  3147.07 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         610.1903    14.6678  41.601  < 2e-16 ***
Log_Freq_HAL         -6.0239     1.9678  -3.061  0.00221 ** 
Length               47.7531     1.6368  29.175  < 2e-16 ***
Log_Freq_HAL:Length  -2.9421     0.2348 -12.528  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 359.1 on 62606 degrees of freedom
Multiple R-squared:  0.09473,	Adjusted R-squared:  0.09469 
F-statistic:  2184 on 3 and 62606 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [8]:
# WRITE YOUR CODE HERE
install.packages("lme4")
library(lme4)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [10]:
# WRITE YOUR CODE HERE
mixed_effects_model <- lmer(D_RT ~ Log_Freq_HAL * Length + (1 | Sub_ID), data = lexical_data)

summary(mixed_effects_model)

Linear mixed model fit by REML ['lmerMod']
Formula: D_RT ~ Log_Freq_HAL * Length + (1 | Sub_ID)
   Data: lexical_data

REML criterion at convergence: 888235.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.5058 -0.5472 -0.1568  0.3103 10.7381 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 46333    215.3   
 Residual             82978    288.1   
Number of obs: 62610, groups:  Sub_ID, 299

Fixed effects:
                    Estimate Std. Error t value
(Intercept)         616.8445    17.1522  35.963
Log_Freq_HAL         -7.4374     1.5830  -4.698
Length               47.7477     1.3162  36.277
Log_Freq_HAL:Length  -2.8778     0.1888 -15.239

Correlation of Fixed Effects:
            (Intr) Lg_F_HAL Length
Log_Frq_HAL -0.645                
Length      -0.656  0.917         
Lg_Fr_HAL:L  0.582 -0.942   -0.923

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> *From the linear model, the t-value is -3.061 for Log_Freq_HAL, 29.175 for length, and -12.528 for Log_Freq_HAL:Length. From the mixed effects model, the t-value is -4.698 for Log_Freq_HAL, 36.277 for length, and -15.239 for Log_Freq_HAL:Length. It appears that all three t-values are slightly larger (in magnitude) from the mixed effects model. This is likely due to the inclusion of the random intercepts. By accounting for natural random variations on the subject-level, the mixed effects model reduces the standard error and subsequently increases the t-values (improves precision).*
>

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [11]:
# WRITE YOUR CODE HERE
AIC(linear_model)
AIC(mixed_effects_model)

[1] 914436.4

[1] 888247.6

> *The mixed effects model has a lower AIC value (888248 instead of 914436), so it is the better one. A lower AIC value suggests that less information was lost or less variance was not explained for.*
>

---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> *Another random effect that could be controlled for is the natural variation between words themselves. Some words may simply be easier to recognize than others for reasons not relevant to length or frequency. In addition, if participants do multiple trials in the study, it is also possible that there are random effects (due to practice or fatigue) occurring when more trials are ran. Thus, controlling for trials could also be insightful.
>

**DUE:** 11:59pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *N/A*

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.
>